[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/20-adaptation.ipynb)

# Module 20: Adaptation: Fine-Tuning, LoRA, Distillation

Companion notebook to [Module 20](https://transformers-and-math.drmhse.com/modules/20-adaptation) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — SVD and the best low-rank approximation (section 20.2)

In [ ]:
W = torch.randn(24, 16)
U, s, Vh = torch.linalg.svd(W, full_matrices=False)
for k in (1, 4, 8, 16):
    Wk = U[:, :k] @ torch.diag(s[:k]) @ Vh[:k]
    print(f'rank {k:2}: error {torch.linalg.norm(W - Wk):.4f}   √(Σ dropped σ²) {s[k:].pow(2).sum().sqrt():.4f}')

## Part 2 — LoRA from scratch (section 20.3)
Pretrain a small network on one task, then adapt it to a related task by training only a rank-r update to each layer.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r, alpha=None):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                       # frozen
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # zero: start exactly at the pretrained model
        self.scale = (alpha or 2 * r) / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

def make_task(seed):
    g = torch.Generator().manual_seed(seed)
    M = torch.randn(32, 32, generator=g)
    return lambda x: torch.tanh(x @ M) @ M.T * 0.1

task_a = make_task(1)
task_b = lambda x: task_a(x) + 0.3 * make_task(2)(x)       # related but different

def fit(model, task, steps=1500, lr=3e-3):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(steps):
        x = torch.randn(128, 32)
        loss = F.mse_loss(model(x), task(x))
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = torch.randn(2048, 32)
        return F.mse_loss(model(x), task(x)).item()

base = nn.Sequential(nn.Linear(32, 128), nn.GELU(), nn.Linear(128, 32))
print('pretrained on task A, loss on A:', round(fit(base, task_a), 5))
base_state = {k: v.clone() for k, v in base.state_dict().items()}

for r in (1, 4, 16):
    m = nn.Sequential(nn.Linear(32, 128), nn.GELU(), nn.Linear(128, 32)); m.load_state_dict(base_state)
    m[0], m[2] = LoRALinear(m[0], r), LoRALinear(m[2], r)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'LoRA rank {r:2}: task-B loss {fit(m, task_b):.5f}   trainable {trainable:,}')
full = nn.Sequential(nn.Linear(32, 128), nn.GELU(), nn.Linear(128, 32)); full.load_state_dict(base_state)
print(f'full fine-tune: task-B loss {fit(full, task_b):.5f}   trainable {sum(p.numel() for p in full.parameters()):,}')

## Part 3 — Distillation (section 20.4)
A small student learns a teacher's soft predictions with the temperature-softened KL loss. Which temperature works best depends on the teacher, the student and the data; this is an experiment to run, not a result to expect.

In [ ]:
teacher = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
with torch.no_grad():
    for p in teacher.parameters():
        p.mul_(3)                                          # a confident teacher

def distill(T, steps=1500):
    torch.manual_seed(0)
    student = nn.Sequential(nn.Linear(20, 16), nn.ReLU(), nn.Linear(16, 10))
    opt = torch.optim.Adam(student.parameters(), lr=3e-3)
    for _ in range(steps):
        x = torch.randn(128, 20)
        with torch.no_grad():
            t = F.softmax(teacher(x) / T, -1)
        loss = T * T * F.kl_div(F.log_softmax(student(x) / T, -1), t, reduction='batchmean')
        opt.zero_grad(); loss.backward(); opt.step()
    x = torch.randn(4096, 20)
    with torch.no_grad():
        return (student(x).argmax(-1) == teacher(x).argmax(-1)).float().mean().item()

for T in (0.5, 1, 2, 4):
    print(f'temperature {T}: student agrees with the teacher on {distill(T):.1%} of inputs')

## Your turn
1. How many trainable parameters does LoRA add to a 4096 × 4096 matrix at rank 8?
2. Initialize `B` randomly instead of at zero. What is the task-A loss of the adapted model before any training?